# LLM-JSON-Extraction

This Jupyter notebook implements a system that extracts structured information from natural language queries about company performance metrics using the Groq API with a language model.

## Key Features

1. **Natural Language Processing**: Extracts structured data from natural language queries
2. **Smart Date Handling**: Sets reasonable defaults for missing dates
3. **History Tracking**: Maintains a record of previous queries
4. **Data Normalization**: Capitalizes names and expands abbreviations

## Use Cases

This notebook could be useful for:
- Financial analysts needing to quickly process and structure queries
- Data visualization tools that require structured input
- Chatbots focused on business intelligence 
- Automated reporting systems

The system takes queries like "Amazon is analysing profits from year 2019-2020" and converts them to structured data that can be used for database queries, visualization, or further analysis.



## 1. Installation
This cell installs the Groq Python client library, which allows the notebook to interact with Groq's API and their language models.

In [ ]:
%pip install groq

## 2. Setup and Configuration
- Imports necessary libraries for API access, date handling, and environment variables
- Uses `load_dotenv()` to load environment variables from a `.env` file
- Creates a Groq client using the API key stored in the environment variables
- Defines a system prompt that instructs the LLM to extract structured information

The system prompt instructs the model to:
- Extract company names, performance metrics, and date ranges
- Format dates in ISO format (YYYY-MM-DD)
- Fix capitalization of company names and metrics
- Expand abbreviations (e.g., GMV → Gross Merchandise Value)
- Return results as valid JSON objects

In [6]:
import os
import json
from groq import Groq
from dotenv import load_dotenv
from datetime import date
from dateutil.relativedelta import relativedelta

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

system_prompt = """You are an assistant that extracts structured information from user queries about company performance metrics.

Extract and return a JSON list of objects with the following fields from the user query:
- entity: Company name (like Amazon, Google, Netflix etc.)
- parameter: Performance metric (like GMV, revenue, profit, etc.)
- start_date: Start date mentioned (or empty string if not specified)
- end_date: End date mentioned (or empty string if not specified)

Choose the date thats smaller as the start date and the larger as the end date. Date should be in ISO format (YYYY-MM-DD).
Auto capitalise and fix case of the company names and performance metrics.
Identify any abbreviations and expand them (e.g., GMV to Gross Merchandise Value).

If the query includes multiple companies or asks for a comparison, return one JSON object per company in the list.

Respond only with valid JSON.
"""


## 3. Query Processing
- Collects a query from the user
- Sends the query to Groq's API using Llama 3.1 8B Instant model
- Extracts the response and attempts to parse it as JSON
- Exits with error code 1 if the response isn't valid JSON


In [7]:
query_input = input("Enter your query: ").strip()

completion = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query_input}
    ]
)

llm_response = completion.choices[0].message.content


try:
    extracted_data = json.loads(llm_response)
except json.JSONDecodeError:
    exit(1)

## 4. History Management and Date Handling
- Gets today's date and calculates a date one year ago
- Specifies a file path for storing query history

The code then:
1. Loads existing history from `history.json` if it exists
2. Creates an empty history if the file doesn't exist or has invalid JSON
3. Processes each extracted data item:
   - Adds it to history
   - Sets default start date to one year ago if not specified
   - Sets default end date to today if not specified
4. Maintains a history of up to 6 items (removes oldest entries)
5. Prints the processed data with filled-in dates
6. Saves the updated history back to `history.json`

In [ ]:
today = date.today()
one_year_ago = today - relativedelta(years=1)

history_file = "history.json"

if os.path.exists(history_file):
    try:
        with open(history_file, "r") as f:
            history = json.load(f)
            if "history" not in history:
                    history["history"] = []
    except json.JSONDecodeError:
        history = {"history": []}
else:
    with open(history_file, "w") as f:
        history = {"history": []}

for i in range(len(extracted_data)):
    history["history"].append(extracted_data[i])
    if(extracted_data[i]["start_date"] == ""):
        extracted_data[i]["start_date"] = one_year_ago.isoformat()
    if(extracted_data[i]["end_date"] == ""):
        extracted_data[i]["end_date"] = today.isoformat()
    if(len(history["history"]) > 6):
        history["history"].pop(0)

    print(extracted_data[i])

with open(history_file, "w") as f:
    json.dump(history, f, indent=2)

print("✅ History updated in history.json")